In [1]:
import ase
import numpy as np

In [2]:
def get_moments_of_inertia(self, vectors=False):
    '''
    Get the moments of inertia along the principal axes.

    The three principal moments of inertia are computed from the
    eigenvalues of the symmetric inertial tensor. Periodic boundary
    conditions are ignored. Units of the moments of inertia are
    amu*angstrom**2.
    '''
    com = self.get_center_of_mass()
    positions = self.get_positions()
    positions -= com  # translate center of mass to origin
    masses = self.get_masses()

    # Initialize elements of the inertial tensor
    I11 = I22 = I33 = I12 = I13 = I23 = 0.0
    for i in range(len(self)):
        x, y, z = positions[i]
        m = masses[i]

        I11 += m * (y ** 2 + z ** 2)
        I22 += m * (x ** 2 + z ** 2)
        I33 += m * (x ** 2 + y ** 2)
        I12 += -m * x * y
        I13 += -m * x * z
        I23 += -m * y * z

    Itensor = np.array([[I11, I12, I13],
                        [I12, I22, I23],
                        [I13, I23, I33]])
    
    Det = np.max(np.abs(Itensor))
    stepen = np.round(np.log10(np.where(Det==0,1,Det)))
    Itensor *= 0.1**stepen
    Itensor = np.round(Itensor,7)*(10**stepen)

    evals, evecs = np.linalg.eig(Itensor)
    if vectors:
        return evals, evecs.transpose()
    else:
        return evals

In [3]:
def Graph_VVS3(atoms):
    '''
    Функция стандартизирует представление структуры путем построения новых базисных векторов 
    и строгого упорядочения атомов.
    ВАЖНО! Для поиска уникальных структур в передоваемом в функцию объекте atoms необходимо
    удолить поглащающий атом!
    '''
    atoms_loc = atoms.copy()
    evals, evecs = get_moments_of_inertia(atoms_loc,vectors=True)
    
    sort_evals_ind = np.argsort(evals)
    evals = evals[sort_evals_ind]
    evecs = evecs[sort_evals_ind]

    
    center_of_mass = atoms_loc.get_center_of_mass()
    pos = atoms_loc.get_positions() - center_of_mass
    atomic_numbers = atoms_loc.get_atomic_numbers()
    N = atomic_numbers.shape[0]
    
    pos_lat = np.linalg.solve(evecs.T, (pos).T).T
    
    atoms_loc.set_positions(pos_lat)
    
    M = atoms_loc.get_all_distances()
    Det = np.linalg.det([np.delete(np.delete(M,i,axis=0), i, axis=1) for i in np.arange(M.shape[0])]).astype(np.float64)
    
    stepen = np.round(np.log10(np.where(Det==0,1,np.abs(Det))))
    Det *= 0.1**stepen
    Det = np.round(Det,5)*(10**stepen)    
    unique,index,inverse,counts = np.unique(np.round(Det,5),return_index=True,return_inverse = True,return_counts=True)

    group_atoms = [np.where(inverse==i)[0] for i in np.arange(np.max(inverse)+1)]
    
    
    global_group_atoms = []
    for i in np.arange(len(group_atoms)):
        group_atom = group_atoms[i]
        if counts[i]>1:
            atom_temp1 = atoms_loc.copy()
            pos1 = atom_temp1.get_positions()
            pos1[group_atom]*=0.9
            atom_temp1.set_positions(pos1)
            M_temp1 = atom_temp1.get_all_distances()
            Det_temp1 = np.linalg.det([np.delete(np.delete(M_temp1,i,axis=0), i, axis=1) for i in group_atom])

            stepen = np.round(np.log10(np.where(Det_temp1==0,1,np.abs(Det_temp1))))
            Det_temp1 *= 0.1**stepen
            Det_temp1 = np.round(Det_temp1,5)*(10**stepen)            
            unique_,index_,inverse_,counts_ = np.unique(np.round(Det_temp1,5),return_index=True,return_inverse = True,return_counts=True)
            group_atoms_loc = [np.where(inverse_==i)[0] for i in np.arange(np.max(inverse_)+1)]
            for group in group_atoms_loc:
                global_group_atoms.append(group_atom[group])
        else:
            global_group_atoms.append(group_atom)
    temp_ind = 0
    

    for i,group in enumerate(global_group_atoms):
        if group.shape[0]>1:
            temp_sort = sorted(zip(np.round(pos_lat[group],5),group), key = lambda x: ((x[0][0]),(x[0][1]),
                                                                                       (x[0][2])))
            global_group_atoms[i] = [j[1] for j in temp_sort]
    
    
    ind_x_pos_lat0 = np.where( np.round(pos_lat[np.concatenate(global_group_atoms),0],5)!=0)[0]
    ind_x_pos_lat0 = 0 if len(ind_x_pos_lat0)==0 else ind_x_pos_lat0[0]    
    ind_y_pos_lat0 = np.where( np.round(pos_lat[np.concatenate(global_group_atoms),1],5)!=0)[0]
    ind_y_pos_lat0 = 0 if len(ind_y_pos_lat0)==0 else ind_y_pos_lat0[0] 
    ind_z_pos_lat0 = np.where( np.round(pos_lat[np.concatenate(global_group_atoms),2],5)!=0)[0]
    ind_z_pos_lat0 = 0 if len(ind_z_pos_lat0)==0 else ind_z_pos_lat0[0] 
    pos_lat0X,pos_lat0Y,pos_lat0Z =  (pos_lat[np.concatenate(global_group_atoms)][ind_x_pos_lat0,0],
    pos_lat[np.concatenate(global_group_atoms)][ind_y_pos_lat0,1],pos_lat[np.concatenate(global_group_atoms)][ind_z_pos_lat0,2])

    pos_lat0 = [pos_lat0X,pos_lat0Y,pos_lat0Z]
    
    if (pos_lat0[0]<0):
        pos_lat[:,0] *= -1
    if (pos_lat0[1]<0):
        pos_lat[:,1] *= -1
    if (pos_lat0[2]<0):
        pos_lat[:,2] *= -1       
    
    for i,group in enumerate(global_group_atoms):
        if np.array(group).shape[0]>1:
            temp_sort = sorted(zip(np.round(pos_lat[group],5),group), key = lambda x: (x[0][0],x[0][1],x[0][2]))
            global_group_atoms[i] = [j[1] for j in temp_sort]
            
            
    group_conc = np.concatenate(global_group_atoms)
    pos_lat_ = pos_lat[group_conc]
    atomic_numbers_ = atomic_numbers[group_conc]
    return np.round(np.column_stack((atomic_numbers_,pos_lat_)),4)
